# Stages 5-7: LAC, APS, and RAPS corruption evaluation

This notebook records clean calibration and CIFAR-10-C evaluation for LAC, APS, and RAPS. It uses the cached Stage 4 probabilities and keeps every clean-calibrated threshold fixed under corruption.

For each fixed CIFAR-10 test split (seeds 42, 43, 44), the 5,000 calibration indices determine one threshold per base model. That threshold is reused on the corresponding 5,000 clean evaluation images and all nine corruption conditions.

Before shifted evaluation, check input integrity, thresholds, and score/set consistency. The three-standard-error coverage screen is a diagnostic that can require investigation. LAC and RAPS enforce a stopping gate; the saved APS run contains one flagged result and a documented post-flag audit and continuation decision below.

## LAC definition and finite-sample threshold

For a probability vector p(x), LAC assigns the candidate label y the nonconformity score

s(x, y) = 1 - p_y(x).

For n calibration examples, sort the true-label calibration scores as s_(1) <= ... <= s_(n). With alpha = 0.10, use

k = ceil((n + 1) * (1 - alpha))

and set q_hat = s_(k), using 1-based order-statistic notation. In code, this is element k-1 after sorting or partitioning. This is the split-conformal finite-sample quantile; it is not NumPy's interpolated percentile. Here n = 5,000 and k = 4,501.

The LAC prediction set is

C(x) = { y : 1 - p_y(x) <= q_hat }.

Calibration always uses the clean calibration indices. The same q_hat remains fixed for clean evaluation and every corrupted condition.

## Inputs and expected checks

Inputs are the 20 Stage 4 prediction caches under `cached_outputs/stage4_predictions/`, the three fixed split files under `data/splits/`, and the tracked CIFAR-10 `test_batch` for canonical labels. The seed-2026 APS/RAPS randomization array is reused from `results/`. Cache rows follow canonical CIFAR-10 test indices 0 through 9,999.

Before calibration, the notebook checks cache shapes, probabilities, canonical indices, labels, split sizes/disjointness, and corruption-cache label alignment. Each model has clean predictions plus Gaussian noise, defocus blur, and brightness at severities 1, 3, and 5.

The clean coverage screen compares each observed coverage with 0.90 using three nominal binomial standard errors: `3 * sqrt(0.90 * 0.10 / 5000)`, approximately 0.0127. This is an investigation trigger, not an exact sampling model for conformal coverage or a test of its marginal guarantee. The saved APS run's exception is described in its clean-data section; it is not counted as a six-of-six screen pass.

In [ ]:
from pathlib import Path
import json
import math
import pickle
import numpy as np
import pandas as pd
from IPython.display import display as show_table

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/conformal_shift")
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = current.parent if current.name == "notebooks" else current

CACHE_ROOT = PROJECT_ROOT / "cached_outputs" / "stage4_predictions"
SPLIT_ROOT = PROJECT_ROOT / "data" / "splits"
CIFAR_TEST_BATCH = PROJECT_ROOT / "data" / "raw" / "cifar-10-batches-py" / "test_batch"
RESULTS_ROOT = PROJECT_ROOT / "results"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

ALPHA = 0.10
TARGET_COVERAGE = 1.0 - ALPHA
SPLIT_SEEDS = (42, 43, 44)
N_TEST = 10_000
EXPECTED_SPLIT_SIZE = 5_000
MODEL_KEYS = ("standard_resnet18", "augmix_resnet18")
CORRUPTIONS = ("gaussian_noise", "defocus_blur", "brightness")
SEVERITIES = (1, 3, 5)
CLASS_NAMES = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache root: {CACHE_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Alpha: {ALPHA:.2f}; target marginal coverage: {TARGET_COVERAGE:.2f}")
print("No model inference is performed in this notebook.")

Mounted at /content/drive
Project root: /content/drive/MyDrive/conformal_shift
Cache root: /content/drive/MyDrive/conformal_shift/cached_outputs/stage4_predictions
Results root: /content/drive/MyDrive/conformal_shift/results
Alpha: 0.10; target marginal coverage: 0.90
No model inference is performed in this notebook.


In [ ]:
# The setup cell above mounts Drive only in Colab and selects the project root.
if IN_COLAB:
    print("Google Drive was mounted by the setup cell.")
else:
    print(f"Local mode: using project files from {PROJECT_ROOT}")


## Load canonical labels, saved splits, and cached predictions

The cached clean labels are compared with the original CIFAR-10 test labels. The split files must contain disjoint calibration/evaluation index sets of 5,000 rows each and together cover all 10,000 canonical test indices.

In [ ]:
if not CIFAR_TEST_BATCH.is_file():
    raise FileNotFoundError(f"Missing CIFAR-10 test batch: {CIFAR_TEST_BATCH}")

with CIFAR_TEST_BATCH.open("rb") as f:
    cifar_test = pickle.load(f, encoding="bytes")
clean_labels = np.asarray(cifar_test[b"labels"], dtype=np.int64)
original_indices = np.arange(N_TEST, dtype=np.int64)
if clean_labels.shape != (N_TEST,):
    raise ValueError(f"Unexpected CIFAR-10 test label shape: {clean_labels.shape}")

splits = {}
for seed in SPLIT_SEEDS:
    split_path = SPLIT_ROOT / f"clean_test_split_seed{seed}.npz"
    with np.load(split_path, allow_pickle=False) as split:
        calibration_indices = np.asarray(split["calibration_indices"], dtype=np.int64)
        evaluation_indices = np.asarray(split["evaluation_indices"], dtype=np.int64)
    combined = np.concatenate([calibration_indices, evaluation_indices])
    if (calibration_indices.shape != (EXPECTED_SPLIT_SIZE,)
            or evaluation_indices.shape != (EXPECTED_SPLIT_SIZE,)
            or np.intersect1d(calibration_indices, evaluation_indices).size
            or np.any(combined < 0) or np.any(combined >= N_TEST)
            or not np.array_equal(np.sort(combined), original_indices)):
        raise ValueError(f"Split seed {seed} failed size, disjointness, bounds, or coverage checks.")
    splits[seed] = {
        "calibration_indices": calibration_indices,
        "evaluation_indices": evaluation_indices,
    }
    print(f"Split {seed}: disjoint 5,000/5,000; covers canonical indices 0..9999: PASS")

CONDITIONS = [("clean", None)] + [
    (corruption, severity)
    for corruption in CORRUPTIONS
    for severity in SEVERITIES
]

def cache_path(model_key, condition, severity):
    filename = "clean_test.npz" if condition == "clean" else f"{condition}_severity_{severity}.npz"
    return CACHE_ROOT / model_key / filename

def load_cache(model_key, condition, severity):
    path = cache_path(model_key, condition, severity)
    if not path.is_file():
        raise FileNotFoundError(f"Missing Stage 4 cache: {path}")
    with np.load(path, allow_pickle=False) as stored:
        cache = {name: stored[name] for name in stored.files}

    logits = np.asarray(cache["logits"])
    probabilities = np.asarray(cache["probabilities"])
    labels = np.asarray(cache["labels"], dtype=np.int64)
    indices = np.asarray(cache["original_test_indices"], dtype=np.int64)
    source_rows = np.asarray(cache["cifar10c_source_rows"], dtype=np.int64)
    metadata = json.loads(str(cache["metadata_json"].item()))

    if logits.shape != (N_TEST, 10) or probabilities.shape != (N_TEST, 10):
        raise ValueError(f"Unexpected output shape in {path}: {logits.shape}, {probabilities.shape}")
    if labels.shape != (N_TEST,) or indices.shape != (N_TEST,) or source_rows.shape != (N_TEST,):
        raise ValueError(f"Index/label lengths do not match in {path}")
    if not np.array_equal(indices, original_indices):
        raise ValueError(f"Canonical indices are not exactly 0..9999 in {path}")
    if not np.isfinite(logits).all() or not np.isfinite(probabilities).all():
        raise ValueError(f"NaN or Inf found in {path}")
    if (np.any(probabilities < 0) or np.any(probabilities > 1)
            or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-5, rtol=1e-5)):
        raise ValueError(f"Invalid probabilities in {path}")
    if not np.array_equal(logits.argmax(axis=1), probabilities.argmax(axis=1)):
        raise ValueError(f"Logit/probability argmax mismatch in {path}")
    if not np.array_equal(labels, clean_labels):
        raise ValueError(f"Cache labels do not align with canonical CIFAR-10 labels in {path}")
    if metadata.get("model_key") != model_key:
        raise ValueError(f"Model metadata mismatch in {path}")
    if metadata.get("condition") != condition or metadata.get("severity") != severity:
        raise ValueError(f"Condition metadata mismatch in {path}")
    if condition == "clean" and not np.all(source_rows == -1):
        raise ValueError(f"Clean cache has unexpected CIFAR-10-C source rows in {path}")
    if condition != "clean":
        lo = (severity - 1) * N_TEST
        hi = severity * N_TEST
        if not np.array_equal(source_rows, np.arange(lo, hi, dtype=np.int64)):
            raise ValueError(f"CIFAR-10-C source-row mismatch in {path}")
    return {
        "logits": logits,
        "probabilities": probabilities,
        "labels": labels,
        "indices": indices,
        "metadata": metadata,
    }

clean_caches = {}
for model_key in MODEL_KEYS:
    clean_caches[model_key] = load_cache(model_key, "clean", None)
    print(f"{model_key}: clean cache shape {clean_caches[model_key]['probabilities'].shape}; label/index checks PASS")

Split 42: disjoint 5,000/5,000; covers canonical indices 0..9999: PASS
Split 43: disjoint 5,000/5,000; covers canonical indices 0..9999: PASS
Split 44: disjoint 5,000/5,000; covers canonical indices 0..9999: PASS
standard_resnet18: clean cache shape (10000, 10); label/index checks PASS
augmix_resnet18: clean cache shape (10000, 10); label/index checks PASS


## Hand-check LAC membership

For the artificial probability vector below, q_hat = 0.70 gives the probability cutoff 1 - q_hat = 0.30. Classes with probability at least 0.30 are included, including the class exactly on the boundary. The expected prediction set is {0, 1}.

In [ ]:
def lac_prediction_sets(probabilities, q_hat):
    probabilities = np.asarray(probabilities)
    if probabilities.ndim != 2:
        raise ValueError("probabilities must have shape (n_examples, n_classes)")
    return (1.0 - probabilities) <= q_hat

toy_probabilities = np.asarray([[0.50, 0.30, 0.15, 0.05]], dtype=np.float64)
toy_q_hat = 0.70
toy_membership = lac_prediction_sets(toy_probabilities, toy_q_hat)
toy_set = np.flatnonzero(toy_membership[0]).tolist()
expected_toy_set = [0, 1]
print("Toy probabilities:", toy_probabilities[0].tolist())
print("Toy q_hat:", toy_q_hat)
print("Computed class indices:", toy_set)
print("Expected class indices:", expected_toy_set)
assert toy_set == expected_toy_set
print("Hand-check example: PASS")

Toy probabilities: [0.5, 0.3, 0.15, 0.05]
Toy q_hat: 0.7
Computed class indices: [0, 1]
Expected class indices: [0, 1]
Hand-check example: PASS


## Calibrate LAC on clean calibration indices

The quantile helper uses the exact finite-sample order statistic with 1-based rank k. The calibration score uses only the probability assigned to each clean calibration example's true class. The six thresholds are model/split-specific calibration results, not separate trained models.

In [ ]:
def finite_sample_lac_threshold(probabilities, labels, calibration_indices, alpha=ALPHA):
    calibration_probabilities = probabilities[calibration_indices]
    calibration_labels = labels[calibration_indices]
    n_calibration = len(calibration_indices)
    k = math.ceil((n_calibration + 1) * (1.0 - alpha))
    if not 1 <= k <= n_calibration:
        raise ValueError(f"Conformal order statistic k={k} is outside 1..n={n_calibration}")
    true_class_probabilities = calibration_probabilities[
        np.arange(n_calibration), calibration_labels
    ]
    scores = 1.0 - true_class_probabilities
    q_hat = float(np.partition(scores, k - 1)[k - 1])
    return q_hat, k, scores

def lac_metrics(probabilities, labels, indices, q_hat):
    selected_probabilities = probabilities[indices]
    selected_labels = labels[indices]
    membership = lac_prediction_sets(selected_probabilities, q_hat)
    set_sizes = membership.sum(axis=1)
    covered = membership[np.arange(len(indices)), selected_labels]
    top1_correct = selected_probabilities.argmax(axis=1) == selected_labels
    return {
        "n": int(len(indices)),
        "coverage": float(covered.mean()),
        "coverage_gap": float(TARGET_COVERAGE - covered.mean()),
        "average_set_size": float(set_sizes.mean()),
        "empty_set_count": int((set_sizes == 0).sum()),
        "empty_set_proportion": float((set_sizes == 0).mean()),
        "top1_accuracy": float(top1_correct.mean()),
        "set_sizes": set_sizes,
        "covered": covered,
        "membership": membership,
    }

threshold_records = []
thresholds = {}
for model_key in MODEL_KEYS:
    clean_cache = clean_caches[model_key]
    for seed in SPLIT_SEEDS:
        q_hat, k, calibration_scores = finite_sample_lac_threshold(
            clean_cache["probabilities"],
            clean_cache["labels"],
            splits[seed]["calibration_indices"],
        )
        thresholds[(model_key, seed)] = q_hat
        threshold_records.append({
            "model_key": model_key,
            "split_seed": seed,
            "n_calibration": EXPECTED_SPLIT_SIZE,
            "alpha": ALPHA,
            "order_statistic_k_1_based": k,
            "q_hat": q_hat,
            "calibration_score_min": float(calibration_scores.min()),
            "calibration_score_median": float(np.median(calibration_scores)),
            "calibration_score_max": float(calibration_scores.max()),
        })

threshold_table = pd.DataFrame(threshold_records)
show_table(threshold_table)
assert len(threshold_table) == 6
assert threshold_table["order_statistic_k_1_based"].eq(4501).all()
print("Six clean calibration thresholds computed; no shifted data used.")

,model_key,split_seed,n_calibration,alpha,order_statistic_k_1_based,q_hat,calibration_score_min,calibration_score_median,calibration_score_max
0,standard_resnet18,42,5000,0.1,4501,0.047375,0.000003,0.000415,0.999988
1,standard_resnet18,43,5000,0.1,4501,0.056374,0.000002,0.000417,0.999988
2,standard_resnet18,44,5000,0.1,4501,0.058355,0.000002,0.000425,0.999988
3,augmix_resnet18,42,5000,0.1,4501,0.156516,0.000108,0.002703,0.999962
4,augmix_resnet18,43,5000,0.1,4501,0.156516,0.000108,0.002750,0.999962
5,augmix_resnet18,44,5000,0.1,4501,0.150586,0.000086,0.002734,0.999962


Six clean calibration thresholds computed; no shifted data used.


## Critical clean-data sanity check

For every model/split pair, evaluate its fixed threshold on that split's clean evaluation indices. Inspect the table before moving to shifted evaluation. If any coverage differs from 0.90 by more than three nominal binomial standard errors, this cell stops with an error; investigate split/cache alignment or threshold implementation before running the corruption section.

The tolerance is about 0.0127 for 5,000 evaluation examples. It is only a debugging trigger; the reported coverage remains the empirical fraction covered.

In [ ]:
clean_records = []
clean_set_size_distribution = []
clean_eval_metrics = {}

for model_key in MODEL_KEYS:
    clean_cache = clean_caches[model_key]
    for seed in SPLIT_SEEDS:
        eval_indices = splits[seed]["evaluation_indices"]
        q_hat = thresholds[(model_key, seed)]
        metrics = lac_metrics(
            clean_cache["probabilities"],
            clean_cache["labels"],
            eval_indices,
            q_hat,
        )
        clean_eval_metrics[(model_key, seed)] = metrics
        clean_records.append({
            "model_key": model_key,
            "split_seed": seed,
            "n_evaluation": metrics["n"],
            "q_hat": q_hat,
            "empirical_coverage": metrics["coverage"],
            "coverage_gap": metrics["coverage_gap"],
            "average_set_size": metrics["average_set_size"],
            "empty_set_count": metrics["empty_set_count"],
            "empty_set_proportion": metrics["empty_set_proportion"],
            "top1_accuracy": metrics["top1_accuracy"],
        })
        counts = np.bincount(metrics["set_sizes"], minlength=11)
        for set_size, count in enumerate(counts):
            clean_set_size_distribution.append({
                "model_key": model_key,
                "split_seed": seed,
                "set_size": set_size,
                "count": int(count),
                "proportion": float(count / metrics["n"]),
            })

clean_table = pd.DataFrame(clean_records)
clean_size_distribution = pd.DataFrame(clean_set_size_distribution)
show_table(clean_table)

sanity_tolerance = 3.0 * math.sqrt(
    TARGET_COVERAGE * (1.0 - TARGET_COVERAGE) / EXPECTED_SPLIT_SIZE
)
print(f"Clean coverage sanity tolerance: ±{sanity_tolerance:.4f} around {TARGET_COVERAGE:.2f}")
bad_clean = clean_table[
    (clean_table["empirical_coverage"] - TARGET_COVERAGE).abs() > sanity_tolerance
]
if not bad_clean.empty:
    show_table(bad_clean)
    raise RuntimeError(
        "Clean LAC coverage is outside the predeclared 3-standard-error range. "
        "Stop and investigate before evaluating corruption."
    )
print("Clean LAC coverage sanity check: PASS for all six model/split combinations.")

print("Clean set-size distribution (set sizes 0 through 10):")
show_table(clean_size_distribution)

,model_key,split_seed,n_evaluation,q_hat,empirical_coverage,coverage_gap,average_set_size,empty_set_count,empty_set_proportion,top1_accuracy
0,standard_resnet18,42,5000,0.047375,0.8938,0.0062,0.9092,454,0.0908,0.9506
1,standard_resnet18,43,5000,0.056374,0.9046,-0.0046,0.9210,395,0.0790,0.9518
2,standard_resnet18,44,5000,0.058355,0.9062,-0.0062,0.9244,378,0.0756,0.9496
3,augmix_resnet18,42,5000,0.156516,0.9024,-0.0024,0.9182,409,0.0818,0.9530
4,augmix_resnet18,43,5000,0.156516,0.9024,-0.0024,0.9184,408,0.0816,0.9530
5,augmix_resnet18,44,5000,0.150586,0.8988,0.0012,0.9156,422,0.0844,0.9522


Clean coverage sanity tolerance: ±0.0127 around 0.90
Clean LAC coverage sanity check: PASS for all six model/split combinations.
Clean set-size distribution (set sizes 0 through 10):


,model_key,split_seed,set_size,count,proportion
0,standard_resnet18,42,0,454,0.0908
1,standard_resnet18,42,1,4546,0.9092
2,standard_resnet18,42,2,0,0.0000
3,standard_resnet18,42,3,0,0.0000
4,standard_resnet18,42,4,0,0.0000
...,...,...,...,...,...
61,augmix_resnet18,44,6,0,0.0000
62,augmix_resnet18,44,7,0,0.0000
63,augmix_resnet18,44,8,0,0.0000
64,augmix_resnet18,44,9,0,0.0000


## Inspect real clean examples and save clean diagnostics

For each model/split, show the first three evaluation indices in saved split order. The table includes the true class, ten predicted probabilities, fixed q_hat, selected class set, and whether the true class is covered. This is an inspectable sample, not a calibration or tuning step.

In [ ]:
example_records = []
for model_key in MODEL_KEYS:
    clean_cache = clean_caches[model_key]
    for seed in SPLIT_SEEDS:
        eval_indices = splits[seed]["evaluation_indices"]
        q_hat = thresholds[(model_key, seed)]
        probabilities = clean_cache["probabilities"][eval_indices[:3]]
        labels = clean_cache["labels"][eval_indices[:3]]
        membership = lac_prediction_sets(probabilities, q_hat)
        for offset, canonical_index in enumerate(eval_indices[:3]):
            prediction_set = np.flatnonzero(membership[offset]).tolist()
            example_records.append({
                "model_key": model_key,
                "split_seed": seed,
                "original_test_index": int(canonical_index),
                "true_label": int(labels[offset]),
                "true_class": CLASS_NAMES[int(labels[offset])],
                "probabilities": np.round(probabilities[offset], 4).tolist(),
                "q_hat": q_hat,
                "prediction_set_indices": prediction_set,
                "prediction_set_classes": [CLASS_NAMES[c] for c in prediction_set],
                "true_label_covered": bool(membership[offset, labels[offset]]),
            })

examples_table = pd.DataFrame(example_records)
show_table(examples_table)

threshold_table.to_csv(RESULTS_ROOT / "lac_calibration_thresholds.csv", index=False)
clean_table.to_csv(RESULTS_ROOT / "lac_clean_metrics.csv", index=False)
clean_size_distribution.to_csv(
    RESULTS_ROOT / "lac_clean_set_size_distribution.csv", index=False
)
examples_table.to_json(
    RESULTS_ROOT / "lac_clean_example_prediction_sets.json",
    orient="records",
    indent=2,
)
print(f"Saved LAC thresholds and clean diagnostics under: {RESULTS_ROOT}")

,model_key,split_seed,original_test_index,true_label,true_class,probabilities,q_hat,prediction_set_indices,prediction_set_classes,true_label_covered
0,standard_resnet18,42,1208,3,cat,"[0.0, 0.0, 0.0006000000284984708, 0.9984999895...",0.047375,[3],[cat],True
1,standard_resnet18,42,3260,7,horse,"[9.999999747378752e-05, 9.999999747378752e-05,...",0.047375,[7],[horse],True
2,standard_resnet18,42,9895,4,deer,"[0.0, 0.0, 0.0, 0.0, 0.9998999834060669, 0.0, ...",0.047375,[4],[deer],True
3,standard_resnet18,43,8518,9,truck,"[0.0006000000284984708, 9.999999747378752e-05,...",0.056374,[9],[truck],True
4,standard_resnet18,43,5136,9,truck,"[0.0, 9.999999747378752e-05, 9.999999747378752...",0.056374,[9],[truck],True
5,standard_resnet18,43,9949,3,cat,"[0.0003000000142492354, 9.999999747378752e-05,...",0.056374,[],[],False
6,standard_resnet18,44,8949,7,horse,"[0.0, 0.0, 0.0, 0.0, 9.999999747378752e-05, 0....",0.058355,[7],[horse],True
7,standard_resnet18,44,7406,7,horse,"[9.999999747378752e-05, 9.999999747378752e-05,...",0.058355,[7],[horse],True
8,standard_resnet18,44,6613,8,ship,"[0.01940000057220459, 9.999999747378752e-05, 0...",0.058355,[8],[ship],True
9,augmix_resnet18,42,1208,3,cat,"[0.0, 0.0, 0.0019000000320374966, 0.9941999912...",0.156516,[3],[cat],True


Saved LAC thresholds and clean diagnostics under: /content/drive/MyDrive/conformal_shift/results


## Evaluate fixed LAC thresholds on approved corruption conditions

This section runs only after the clean sanity check passes. For each model, split, corruption, and severity, it indexes the existing Stage 4 cache at the saved evaluation indices and applies the already calibrated clean threshold. It never computes a new threshold from corrupted examples.

Reported metrics are empirical coverage, coverage gap (0.90 - coverage), average prediction-set size, empty-set count, and ordinary top-1 accuracy from cached probabilities.

In [ ]:
corruption_records = []
for model_key in MODEL_KEYS:
    for seed in SPLIT_SEEDS:
        eval_indices = splits[seed]["evaluation_indices"]
        q_hat = thresholds[(model_key, seed)]
        for corruption in CORRUPTIONS:
            for severity in SEVERITIES:
                cache = load_cache(model_key, corruption, severity)
                metrics = lac_metrics(
                    cache["probabilities"],
                    cache["labels"],
                    eval_indices,
                    q_hat,
                )
                corruption_records.append({
                    "model_key": model_key,
                    "split_seed": seed,
                    "condition": corruption,
                    "severity": severity,
                    "n_evaluation": metrics["n"],
                    "fixed_clean_q_hat": q_hat,
                    "empirical_coverage": metrics["coverage"],
                    "coverage_gap": metrics["coverage_gap"],
                    "average_set_size": metrics["average_set_size"],
                    "empty_set_count": metrics["empty_set_count"],
                    "empty_set_proportion": metrics["empty_set_proportion"],
                    "top1_accuracy": metrics["top1_accuracy"],
                })

corruption_table = pd.DataFrame(corruption_records)
if len(corruption_table) != 2 * 3 * 3 * 3:
    raise ValueError(f"Expected 54 model/split/condition rows, found {len(corruption_table)}")
show_table(corruption_table)
corruption_table.to_csv(RESULTS_ROOT / "lac_corruption_metrics.csv", index=False)
print(f"Saved corruption metrics under: {RESULTS_ROOT / 'lac_corruption_metrics.csv'}")

,model_key,split_seed,condition,severity,n_evaluation,fixed_clean_q_hat,empirical_coverage,coverage_gap,average_set_size,empty_set_count,empty_set_proportion,top1_accuracy
0,standard_resnet18,42,gaussian_noise,1,5000,0.047375,0.6668,0.2332,0.7316,1342,0.2684,0.8012
1,standard_resnet18,42,gaussian_noise,3,5000,0.047375,0.2626,0.6374,0.4696,2652,0.5304,0.4104
2,standard_resnet18,42,gaussian_noise,5,5000,0.047375,0.1646,0.7354,0.4482,2759,0.5518,0.2686
3,standard_resnet18,42,defocus_blur,1,5000,0.047375,0.8966,0.0034,0.9138,431,0.0862,0.9496
4,standard_resnet18,42,defocus_blur,3,5000,0.047375,0.7872,0.1128,0.8218,891,0.1782,0.8890
5,standard_resnet18,42,defocus_blur,5,5000,0.047375,0.3740,0.5260,0.4886,2557,0.5114,0.5872
6,standard_resnet18,42,brightness,1,5000,0.047375,0.8932,0.0068,0.9088,456,0.0912,0.9488
7,standard_resnet18,42,brightness,3,5000,0.047375,0.8804,0.0196,0.8990,505,0.1010,0.9398
8,standard_resnet18,42,brightness,5,5000,0.047375,0.8356,0.0644,0.8652,674,0.1348,0.9138
9,standard_resnet18,43,gaussian_noise,1,5000,0.056374,0.6698,0.2302,0.7384,1308,0.2616,0.7986


Saved corruption metrics under: /content/drive/MyDrive/conformal_shift/results/lac_corruption_metrics.csv


In [ ]:
audit_thresholds = threshold_table.copy()
audit_thresholds["probability_cutoff"] = 1.0 - audit_thresholds["q_hat"]
print("LAC thresholds and equivalent probability cutoffs:")
show_table(audit_thresholds[["model_key", "split_seed", "q_hat", "probability_cutoff"]])

print("Clean coverage and average set size:")
show_table(clean_table[["model_key", "split_seed", "empirical_coverage", "average_set_size"]])

size_audit = clean_size_distribution.copy()
size_audit["size_bin"] = np.where(size_audit["set_size"] >= 3, ">=3", size_audit["set_size"].astype(str))
size_audit = (size_audit.groupby(["model_key", "split_seed", "size_bin"], as_index=False)["count"].sum()
              .pivot(index=["model_key", "split_seed"], columns="size_bin", values="count")
              .fillna(0).astype(int).reset_index())
for bucket in ["0", "1", "2", ">=3"]:
    if bucket not in size_audit:
        size_audit[bucket] = 0
size_audit = size_audit[["model_key", "split_seed", "0", "1", "2", ">=3"]]
print("Clean prediction-set size counts (0, 1, 2, >=3):")
show_table(size_audit)

expected_q = {(r.model_key, int(r.split_seed)): float(r.q_hat) for r in threshold_table.itertuples()}
corruption_q_matches = all(
    np.isclose(float(row.fixed_clean_q_hat), expected_q[(row.model_key, int(row.split_seed))])
    for row in corruption_table.itertuples()
)
split_counts = corruption_table.groupby("model_key")["split_seed"].nunique()
assert len(corruption_table) == 54 and (split_counts == 3).all() and corruption_q_matches
print("Corruption table audit: 54 rows; three split-specific evaluations per condition/model; every fixed_clean_q_hat matches its clean calibration threshold: PASS")

LAC thresholds and equivalent probability cutoffs:


,model_key,split_seed,q_hat,probability_cutoff
0,standard_resnet18,42,0.047375,0.952625
1,standard_resnet18,43,0.056374,0.943626
2,standard_resnet18,44,0.058355,0.941645
3,augmix_resnet18,42,0.156516,0.843484
4,augmix_resnet18,43,0.156516,0.843484
5,augmix_resnet18,44,0.150586,0.849414


Clean coverage and average set size:


,model_key,split_seed,empirical_coverage,average_set_size
0,standard_resnet18,42,0.8938,0.9092
1,standard_resnet18,43,0.9046,0.9210
2,standard_resnet18,44,0.9062,0.9244
3,augmix_resnet18,42,0.9024,0.9182
4,augmix_resnet18,43,0.9024,0.9184
5,augmix_resnet18,44,0.8988,0.9156


Clean prediction-set size counts (0, 1, 2, >=3):


size_bin,model_key,split_seed,0,1,2,>=3
0,augmix_resnet18,42,409,4591,0,0
1,augmix_resnet18,43,408,4592,0,0
2,augmix_resnet18,44,422,4578,0,0
3,standard_resnet18,42,454,4546,0,0
4,standard_resnet18,43,395,4605,0,0
5,standard_resnet18,44,378,4622,0,0


Corruption table audit: 54 rows; three split-specific evaluations per condition/model; every fixed_clean_q_hat matches its clean calibration threshold: PASS


# Stage 6 — APS calibration and clean validation

## APS score and randomization

Sort each image's class probabilities from largest to smallest. If candidate label y has rank r and probability p_(r), its randomized APS nonconformity score is

**s_APS(x, y; U) = sum_{j < r} p_(j)(x) + U · p_(r)(x), where U ~ Uniform(0,1).**

The first term is the probability mass assigned to classes ranked ahead of y. U randomizes how much of y's own probability mass is included. One U is used for all candidate labels for an image, so included labels stay a prefix of the probability ranking. Stable sorting resolves equal probabilities by class index.

Calibration uses the true-label scores on clean calibration images. With n calibration examples, use k = ceil((n + 1) · (1 − alpha)) and the k-th ordered score (1-based). Prediction sets include candidates whose randomized cumulative score is at most q_hat.

A fixed RNG seed generates one U for each canonical CIFAR-10 test index for reproducibility. Within each split, calibration and evaluation indices are disjoint. This section uses the existing cached probabilities only; it does not run inference, tune on shifted data, or implement RAPS.

In [ ]:
APS_RANDOM_SEED = 2026
APS_UNIFORMS_PATH = RESULTS_ROOT / f"aps_randomization_uniforms_seed{APS_RANDOM_SEED}.npy"
regenerated_uniforms = np.random.default_rng(APS_RANDOM_SEED).random(N_TEST)
if APS_UNIFORMS_PATH.exists():
    aps_uniform_by_index = np.load(APS_UNIFORMS_PATH, allow_pickle=False)
    if aps_uniform_by_index.shape != (N_TEST,) or not np.array_equal(aps_uniform_by_index, regenerated_uniforms):
        raise ValueError("Saved APS randomization uniforms do not match the fixed seed and canonical index mapping")
else:
    aps_uniform_by_index = regenerated_uniforms
    np.save(APS_UNIFORMS_PATH, aps_uniform_by_index, allow_pickle=False)
assert np.all((aps_uniform_by_index >= 0) & (aps_uniform_by_index < 1))
print(f"APS randomization: one reproducible U per canonical test index; seed={APS_RANDOM_SEED}; saved at {APS_UNIFORMS_PATH}")
print("Uniforms are assigned by canonical image index only, independent of label and corruption condition.")

def aps_nonconformity_scores(probabilities, labels, uniforms):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    uniforms = np.asarray(uniforms, dtype=np.float64)
    if probabilities.ndim != 2 or labels.shape != (len(probabilities),) or uniforms.shape != (len(probabilities),):
        raise ValueError("Expected probabilities (N,C), labels (N,), and uniforms (N,)")
    if np.any(labels < 0) or np.any(labels >= probabilities.shape[1]):
        raise ValueError("Labels are outside the probability class range")
    if np.any(uniforms < 0) or np.any(uniforms >= 1):
        raise ValueError("APS randomization values must lie in [0,1)")
    order = np.argsort(-probabilities, axis=1, kind="stable")
    sorted_probabilities = np.take_along_axis(probabilities, order, axis=1)
    sorted_before = np.cumsum(sorted_probabilities, axis=1) - sorted_probabilities
    ranks = np.argmax(order == labels[:, None], axis=1)
    rows = np.arange(len(labels))
    return sorted_before[rows, ranks] + uniforms * probabilities[rows, labels]

def aps_prediction_sets(probabilities, q_hat, uniforms):
    probabilities = np.asarray(probabilities, dtype=np.float64)
    uniforms = np.asarray(uniforms, dtype=np.float64)
    if probabilities.ndim != 2 or uniforms.shape != (len(probabilities),):
        raise ValueError("Expected probabilities (N,C) and uniforms (N,)")
    if np.any(uniforms < 0) or np.any(uniforms >= 1):
        raise ValueError("APS randomization values must lie in [0,1)")
    order = np.argsort(-probabilities, axis=1, kind="stable")
    sorted_probabilities = np.take_along_axis(probabilities, order, axis=1)
    sorted_before = np.cumsum(sorted_probabilities, axis=1) - sorted_probabilities
    included_in_rank_order = sorted_before + uniforms[:, None] * sorted_probabilities <= q_hat
    if np.any((~included_in_rank_order[:, :-1]) & included_in_rank_order[:, 1:]):
        raise AssertionError("APS membership must be a prefix of sorted class probabilities")
    membership = np.zeros_like(included_in_rank_order, dtype=bool)
    np.put_along_axis(membership, order, included_in_rank_order, axis=1)
    return membership

# Hand-check: with U=0.5, scores are 0.25, 0.65, 0.875, 0.975.
# At q_hat=0.65, the boundary is included, so the prediction set is {0,1}.
aps_toy_probabilities = np.asarray([[0.50, 0.30, 0.15, 0.05]])
aps_toy_uniforms = np.asarray([0.50])
aps_toy_scores = aps_nonconformity_scores(aps_toy_probabilities, np.asarray([0]), aps_toy_uniforms)
aps_toy_membership = aps_prediction_sets(aps_toy_probabilities, 0.65, aps_toy_uniforms)
aps_toy_set = np.flatnonzero(aps_toy_membership[0]).tolist()
assert aps_toy_set == [0, 1]
assert np.isclose(aps_toy_scores[0], 0.25)
print("APS hand-check scores [0.25, 0.65, 0.875, 0.975]; q_hat=0.65; set [0, 1]: PASS")

APS randomization: one reproducible U per canonical test index; seed=2026; saved at /content/drive/MyDrive/conformal_shift/results/aps_randomization_uniforms_seed2026.npy
Uniforms are assigned by canonical image index only, independent of label and corruption condition.
APS hand-check scores [0.25, 0.65, 0.875, 0.975]; q_hat=0.65; set [0, 1]: PASS


## Calibrate APS on clean calibration data

For each model and saved split, score the true labels only on that split's clean calibration indices. With n=5,000 and alpha=0.10, k=ceil((n+1)(1-alpha))=4,501; the threshold is the 4,501st ordered randomized APS score. This produces six model/split-specific thresholds. The next cell shows those thresholds before clean evaluation.

In [ ]:
aps_threshold_records = []
aps_thresholds = {}
for model_key in MODEL_KEYS:
    probabilities = clean_caches[model_key]["probabilities"]
    labels = clean_caches[model_key]["labels"]
    for seed in SPLIT_SEEDS:
        calibration_indices = splits[seed]["calibration_indices"]
        scores = aps_nonconformity_scores(
            probabilities[calibration_indices], labels[calibration_indices],
            aps_uniform_by_index[calibration_indices],
        )
        n_calibration = len(calibration_indices)
        k = math.ceil((n_calibration + 1) * (1.0 - ALPHA))
        if not 1 <= k <= n_calibration:
            raise ValueError(f"APS order statistic k={k} is outside 1..n={n_calibration}")
        q_hat = float(np.partition(scores, k - 1)[k - 1])
        aps_thresholds[(model_key, seed)] = q_hat
        aps_threshold_records.append({
            "model_key": model_key, "split_seed": seed,
            "n_calibration": n_calibration, "alpha": ALPHA,
            "order_statistic_k_1_based": k, "q_hat": q_hat,
            "calibration_score_min": float(scores.min()),
            "calibration_score_median": float(np.median(scores)),
            "calibration_score_max": float(scores.max()),
        })
aps_threshold_table = pd.DataFrame(aps_threshold_records)
assert len(aps_threshold_table) == 6
print("Six clean-calibrated APS thresholds (not six trained models):")
show_table(aps_threshold_table)

Six clean-calibrated APS thresholds (not six trained models):


,model_key,split_seed,n_calibration,alpha,order_statistic_k_1_based,q_hat,calibration_score_min,calibration_score_median,calibration_score_max
0,standard_resnet18,42,5000,0.1,4501,0.925118,0.000184,0.521197,0.999978
1,standard_resnet18,43,5000,0.1,4501,0.927148,0.000184,0.528827,0.999978
2,standard_resnet18,44,5000,0.1,4501,0.928890,0.000184,0.530421,0.999978
3,augmix_resnet18,42,5000,0.1,4501,0.909606,0.000181,0.506161,0.999888
4,augmix_resnet18,43,5000,0.1,4501,0.912228,0.000181,0.508366,0.999916
5,augmix_resnet18,44,5000,0.1,4501,0.919139,0.000181,0.517384,0.999888


### APS clean-data sanity check and recorded exception

Apply each clean-calibrated threshold to its disjoint clean evaluation indices. The original workflow called for stopping to investigate coverage more than three nominal standard errors from 0.90. The saved APS diagnostic reports **5/6 within the 0.0127 tolerance**: AugMix with split seed 44 has coverage **0.9140**, a deviation of +0.0140 (about 3.30 nominal standard errors).

The recorded run continued after a separate consistency audit: all six calibration order statistics were reproduced, and true-label score-threshold membership agreed with prediction-set membership for all 30,000 clean evaluation rows. The input checks also verified canonical labels and disjoint splits. Those checks support the implementation calculation; they do not make the flagged result pass the original coverage screen.

This was a post-flag diagnostic continuation, rather than compliance with the originally stated stopping rule. The threshold, randomization seed, split indices, tolerance, and empirical coverage were retained. The saved CSVs and displayed outputs still report 0.9140. The audit runs before the persistence cell below, and corruption evaluation uses the same clean-calibrated thresholds.

In [ ]:
aps_clean_records = []
aps_clean_size_records = []
aps_clean_example_records = []
for model_key in MODEL_KEYS:
    cache = clean_caches[model_key]
    probabilities = cache["probabilities"]
    labels = cache["labels"]
    for seed in SPLIT_SEEDS:
        eval_indices = splits[seed]["evaluation_indices"]
        q_hat = aps_thresholds[(model_key, seed)]
        membership = aps_prediction_sets(probabilities[eval_indices], q_hat, aps_uniform_by_index[eval_indices])
        set_sizes = membership.sum(axis=1)
        covered = membership[np.arange(len(eval_indices)), labels[eval_indices]]
        coverage = float(covered.mean())
        aps_clean_records.append({
            "model_key": model_key, "split_seed": seed, "n_evaluation": len(eval_indices),
            "q_hat": q_hat, "empirical_coverage": coverage,
            "coverage_gap": TARGET_COVERAGE - coverage,
            "average_set_size": float(set_sizes.mean()),
            "empty_set_count": int((set_sizes == 0).sum()),
            "empty_set_proportion": float((set_sizes == 0).mean()),
            "top1_accuracy": float((probabilities[eval_indices].argmax(axis=1) == labels[eval_indices]).mean()),
        })
        for size in range(probabilities.shape[1] + 1):
            aps_clean_size_records.append({
                "model_key": model_key, "split_seed": seed, "set_size": size,
                "count": int((set_sizes == size).sum()), "proportion": float((set_sizes == size).mean()),
            })
        for offset, canonical_index in enumerate(eval_indices[:3]):
            prediction_set = np.flatnonzero(membership[offset]).tolist()
            aps_clean_example_records.append({
                "model_key": model_key, "split_seed": seed,
                "original_test_index": int(canonical_index),
                "true_label": int(labels[canonical_index]),
                "true_class": CLASS_NAMES[int(labels[canonical_index])],
                "probabilities": np.round(probabilities[canonical_index], 4).tolist(),
                "uniform_U": float(aps_uniform_by_index[canonical_index]),
                "q_hat": q_hat, "prediction_set_indices": prediction_set,
                "prediction_set_classes": [CLASS_NAMES[c] for c in prediction_set],
                "true_label_covered": bool(membership[offset, labels[canonical_index]]),
            })
aps_clean_table = pd.DataFrame(aps_clean_records)
aps_clean_size_distribution = pd.DataFrame(aps_clean_size_records)
aps_clean_examples = pd.DataFrame(aps_clean_example_records)
aps_clean_tolerance = 3 * math.sqrt(TARGET_COVERAGE * (1 - TARGET_COVERAGE) / EXPECTED_SPLIT_SIZE)
print(f"Clean APS coverage diagnostic tolerance: +/-{aps_clean_tolerance:.4f} around {TARGET_COVERAGE:.2f}")
show_table(aps_clean_table)
print("Clean prediction-set size distributions:")
show_table(aps_clean_size_distribution)
print("Representative clean examples (3 per model/split):")
show_table(aps_clean_examples)
within_three_se = np.abs(aps_clean_table["empirical_coverage"] - TARGET_COVERAGE) <= aps_clean_tolerance
print(f"Three-SE diagnostic: {int(within_three_se.sum())}/6 combinations within +/-{aps_clean_tolerance:.4f}.")
print(f"Maximum absolute coverage deviation: {np.abs(aps_clean_table['empirical_coverage'] - TARGET_COVERAGE).max():.4f}.")

Clean APS coverage diagnostic tolerance: +/-0.0127 around 0.90


,model_key,split_seed,n_evaluation,q_hat,empirical_coverage,coverage_gap,average_set_size,empty_set_count,empty_set_proportion,top1_accuracy
0,standard_resnet18,42,5000,0.925118,0.8970,0.0030,0.9906,363,0.0726,0.9506
1,standard_resnet18,43,5000,0.927148,0.9012,-0.0012,0.9938,339,0.0678,0.9518
2,standard_resnet18,44,5000,0.928890,0.9068,-0.0068,0.9992,301,0.0602,0.9496
3,augmix_resnet18,42,5000,0.909606,0.8946,0.0054,1.0192,413,0.0826,0.9530
4,augmix_resnet18,43,5000,0.912228,0.8988,0.0012,1.0244,389,0.0778,0.9530
5,augmix_resnet18,44,5000,0.919139,0.9140,-0.0140,1.0456,326,0.0652,0.9522


Clean prediction-set size distributions:


,model_key,split_seed,set_size,count,proportion
0,standard_resnet18,42,0,363,0.0726
1,standard_resnet18,42,1,4386,0.8772
2,standard_resnet18,42,2,195,0.0390
3,standard_resnet18,42,3,48,0.0096
4,standard_resnet18,42,4,7,0.0014
...,...,...,...,...,...
61,augmix_resnet18,44,6,1,0.0002
62,augmix_resnet18,44,7,0,0.0000
63,augmix_resnet18,44,8,0,0.0000
64,augmix_resnet18,44,9,0,0.0000


Representative clean examples (3 per model/split):


,model_key,split_seed,original_test_index,true_label,true_class,probabilities,uniform_U,q_hat,prediction_set_indices,prediction_set_classes,true_label_covered
0,standard_resnet18,42,1208,3,cat,"[0.0, 0.0, 0.0006000000284984708, 0.9984999895...",0.312702,0.925118,[3],[cat],True
1,standard_resnet18,42,3260,7,horse,"[9.999999747378752e-05, 9.999999747378752e-05,...",0.073377,0.925118,[7],[horse],True
2,standard_resnet18,42,9895,4,deer,"[0.0, 0.0, 0.0, 0.0, 0.9998999834060669, 0.0, ...",0.241804,0.925118,[4],[deer],True
3,standard_resnet18,43,8518,9,truck,"[0.0006000000284984708, 9.999999747378752e-05,...",0.317131,0.927148,[9],[truck],True
4,standard_resnet18,43,5136,9,truck,"[0.0, 9.999999747378752e-05, 9.999999747378752...",0.060954,0.927148,[9],[truck],True
5,standard_resnet18,43,9949,3,cat,"[0.0003000000142492354, 9.999999747378752e-05,...",0.880171,0.927148,[3],[cat],True
6,standard_resnet18,44,8949,7,horse,"[0.0, 0.0, 0.0, 0.0, 9.999999747378752e-05, 0....",0.206038,0.928890,[7],[horse],True
7,standard_resnet18,44,7406,7,horse,"[9.999999747378752e-05, 9.999999747378752e-05,...",0.423080,0.928890,[7],[horse],True
8,standard_resnet18,44,6613,8,ship,"[0.01940000057220459, 9.999999747378752e-05, 0...",0.951385,0.928890,[],[],False
9,augmix_resnet18,42,1208,3,cat,"[0.0, 0.0, 0.0019000000320374966, 0.9941999912...",0.312702,0.909606,[3],[cat],True


Three-SE diagnostic: 5/6 combinations within +/-0.0127.
Maximum absolute coverage deviation: 0.0140.


In [ ]:
aps_consistency_records = []
for model_key in MODEL_KEYS:
    probabilities = clean_caches[model_key]["probabilities"]
    labels = clean_caches[model_key]["labels"]
    for seed in SPLIT_SEEDS:
        cal_idx = splits[seed]["calibration_indices"]
        eval_idx = splits[seed]["evaluation_indices"]
        q_hat = aps_thresholds[(model_key, seed)]
        cal_scores = aps_nonconformity_scores(
            probabilities[cal_idx], labels[cal_idx], aps_uniform_by_index[cal_idx]
        )
        k = math.ceil((len(cal_idx) + 1) * (1.0 - ALPHA))
        recomputed_q = float(np.partition(cal_scores, k - 1)[k - 1])
        eval_scores = aps_nonconformity_scores(
            probabilities[eval_idx], labels[eval_idx], aps_uniform_by_index[eval_idx]
        )
        predicted_membership = aps_prediction_sets(
            probabilities[eval_idx], q_hat, aps_uniform_by_index[eval_idx]
        )
        set_coverage = predicted_membership[np.arange(len(eval_idx)), labels[eval_idx]]
        score_coverage = eval_scores <= q_hat
        assert np.array_equal(set_coverage, score_coverage), (
            f"True-label set membership differs from APS score threshold: {model_key}, seed {seed}"
        )
        assert np.isclose(q_hat, recomputed_q), f"Calibration quantile mismatch: {model_key}, seed {seed}"
        coverage = float(set_coverage.mean())
        aps_consistency_records.append({
            "model_key": model_key, "split_seed": seed,
            "coverage_from_set_membership": coverage,
            "coverage_from_true_score_le_q": float(score_coverage.mean()),
            "difference_from_target": coverage - TARGET_COVERAGE,
            "nominal_standard_errors": (coverage - TARGET_COVERAGE) / math.sqrt(TARGET_COVERAGE * (1-TARGET_COVERAGE) / len(eval_idx)),
            "q_hat_recomputed_matches": True,
            "set_and_score_coverage_match": True,
        })

aps_consistency_table = pd.DataFrame(aps_consistency_records)
show_table(aps_consistency_table)
print("APS audit: all six calibration order statistics recompute exactly; true-label set membership equals score <= q_hat for all 30,000 clean evaluation rows: PASS.")
print("The 3-SE flag was a diagnostic threshold, not a conformal validity condition. Inspect the small observed deviation in context across six split/model comparisons.")
# Save the clean APS outputs after confirming the scoring and membership implementations agree.
aps_threshold_table.to_csv(RESULTS_ROOT / "aps_calibration_thresholds.csv", index=False)
aps_clean_table.to_csv(RESULTS_ROOT / "aps_clean_metrics.csv", index=False)
aps_clean_size_distribution.to_csv(RESULTS_ROOT / "aps_clean_set_size_distribution.csv", index=False)
aps_clean_examples.to_json(RESULTS_ROOT / "aps_clean_example_prediction_sets.json", orient="records", indent=2)
aps_consistency_table.to_csv(RESULTS_ROOT / "aps_clean_consistency_audit.csv", index=False)
print(f"Saved APS clean calibration, metrics, set-size distribution, examples, and score/set audit under: {RESULTS_ROOT}")

,model_key,split_seed,coverage_from_set_membership,coverage_from_true_score_le_q,difference_from_target,nominal_standard_errors,q_hat_recomputed_matches,set_and_score_coverage_match
0,standard_resnet18,42,0.8970,0.8970,-0.0030,-0.707107,True,True
1,standard_resnet18,43,0.9012,0.9012,0.0012,0.282843,True,True
2,standard_resnet18,44,0.9068,0.9068,0.0068,1.602775,True,True
3,augmix_resnet18,42,0.8946,0.8946,-0.0054,-1.272792,True,True
4,augmix_resnet18,43,0.8988,0.8988,-0.0012,-0.282843,True,True
5,augmix_resnet18,44,0.9140,0.9140,0.0140,3.299832,True,True


APS audit: all six calibration order statistics recompute exactly; true-label set membership equals score <= q_hat for all 30,000 clean evaluation rows: PASS.
The 3-SE flag was a diagnostic threshold, not a conformal validity condition. Inspect the small observed deviation in context across six split/model comparisons.
Saved APS clean calibration, metrics, set-size distribution, examples, and score/set audit under: /content/drive/MyDrive/conformal_shift/results


In [ ]:
# Persist APS clean diagnostics after the independent score/set consistency audit.
assert len(aps_threshold_table) == 6
assert len(aps_clean_table) == 6
assert len(aps_clean_size_distribution) == 66
assert len(aps_clean_examples) == 18
aps_threshold_table.to_csv(RESULTS_ROOT / "aps_calibration_thresholds.csv", index=False)
aps_clean_table.to_csv(RESULTS_ROOT / "aps_clean_metrics.csv", index=False)
aps_clean_size_distribution.to_csv(RESULTS_ROOT / "aps_clean_set_size_distribution.csv", index=False)
aps_clean_examples.to_json(RESULTS_ROOT / "aps_clean_example_prediction_sets.json", orient="records", indent=2)
aps_consistency_table.to_csv(RESULTS_ROOT / "aps_clean_consistency_audit.csv", index=False)
aps_saved_metrics = pd.read_csv(RESULTS_ROOT / "aps_clean_metrics.csv")
assert len(aps_saved_metrics) == 6
assert np.allclose(aps_saved_metrics["empirical_coverage"], aps_clean_table["empirical_coverage"])
assert np.allclose(aps_saved_metrics["average_set_size"], aps_clean_table["average_set_size"])
print("APS clean files saved; six metric rows reloaded and match the in-memory APS results: PASS")
print(aps_saved_metrics[["model_key", "split_seed", "q_hat", "empirical_coverage", "average_set_size", "empty_set_count"]].to_string(index=False))

APS clean files saved; six metric rows reloaded and match the in-memory APS results: PASS
        model_key  split_seed    q_hat  empirical_coverage  average_set_size  empty_set_count
standard_resnet18          42 0.925118              0.8970            0.9906              363
standard_resnet18          43 0.927148              0.9012            0.9938              339
standard_resnet18          44 0.928890              0.9068            0.9992              301
  augmix_resnet18          42 0.909606              0.8946            1.0192              413
  augmix_resnet18          43 0.912228              0.8988            1.0244              389
  augmix_resnet18          44 0.919139              0.9140            1.0456              326


## Evaluate APS with the fixed clean thresholds

For each base model and saved split, this section applies the threshold calibrated on clean calibration data to the corresponding evaluation images in the nine approved CIFAR-10-C conditions. The calibration thresholds stay fixed. No network inference or recalibration is performed.

Each canonical CIFAR-10 test image uses one saved uniform random value. The same value is used for its clean and corrupted versions and across candidate labels, enabling paired comparisons. The cell reports split-specific coverage, coverage gap, average set size, empty-set rate, and top-1 accuracy, plus the full distribution of set sizes. It joins APS and LAC on model, split, corruption, and severity, then summarizes the three splits with mean and sample standard deviation. Expected checks: 54 APS rows, 594 set-size rows, all six clean thresholds retained, and a 54-row one-to-one LAC comparison.

In [ ]:
# Reproducibility audit and APS corruption evaluation from cached probabilities only.
saved_uniforms = np.load(APS_UNIFORMS_PATH, allow_pickle=False)
regenerated_uniforms = np.random.default_rng(APS_RANDOM_SEED).random(N_TEST)
assert saved_uniforms.shape == (N_TEST,) and np.array_equal(saved_uniforms, regenerated_uniforms)
assert np.array_equal(aps_uniform_by_index, saved_uniforms)
assert np.all((saved_uniforms >= 0) & (saved_uniforms < 1))
print("Randomization audit: saved uniforms reproduce exactly from seed 2026; one U is keyed by canonical index, independent of label/severity: PASS")
aps_corruption_records, aps_corruption_size_records = [], []
for model_key in MODEL_KEYS:
    labels = clean_caches[model_key]["labels"]
    for seed in SPLIT_SEEDS:
        eval_indices = splits[seed]["evaluation_indices"]
        q_hat = aps_thresholds[(model_key, seed)]
        uniforms = saved_uniforms[eval_indices]
        for corruption in CORRUPTIONS:
            for severity in SEVERITIES:
                cache = load_cache(model_key, corruption, severity)
                assert np.array_equal(cache["labels"], labels)
                probabilities = cache["probabilities"][eval_indices]
                membership = aps_prediction_sets(probabilities, q_hat, uniforms)
                sizes = membership.sum(axis=1)
                covered = membership[np.arange(len(eval_indices)), labels[eval_indices]]
                aps_corruption_records.append({
                    "model_key": model_key, "split_seed": seed, "condition": corruption, "severity": int(severity),
                    "n_evaluation": len(eval_indices), "fixed_clean_q_hat": q_hat,
                    "empirical_coverage": float(covered.mean()), "coverage_gap": TARGET_COVERAGE - float(covered.mean()),
                    "average_set_size": float(sizes.mean()), "empty_set_count": int((sizes == 0).sum()),
                    "empty_set_proportion": float((sizes == 0).mean()),
                    "top1_accuracy": float((probabilities.argmax(axis=1) == labels[eval_indices]).mean()),
                })
                for size in range(11):
                    aps_corruption_size_records.append({
                        "model_key": model_key, "split_seed": seed, "condition": corruption, "severity": int(severity),
                        "set_size": size, "count": int((sizes == size).sum()), "proportion": float((sizes == size).mean()),
                    })
aps_corruption_table = pd.DataFrame(aps_corruption_records)
aps_corruption_size_distribution = pd.DataFrame(aps_corruption_size_records)
assert len(aps_corruption_table) == 54 and len(aps_corruption_size_distribution) == 594
expected_q = {(r.model_key, int(r.split_seed)): float(r.q_hat) for r in aps_threshold_table.itertuples()}
assert all(np.isclose(r.fixed_clean_q_hat, expected_q[(r.model_key, int(r.split_seed))]) for r in aps_corruption_table.itertuples())
assert (aps_corruption_table.groupby(["model_key", "condition", "severity"]).size() == 3).all()
lac_table = pd.read_csv(RESULTS_ROOT / "lac_corruption_metrics.csv")
keys = ["model_key", "split_seed", "condition", "severity"]
assert len(lac_table) == 54
paired = aps_corruption_table.merge(lac_table, on=keys, suffixes=("_aps", "_lac"), validate="one_to_one")
assert len(paired) == 54
for m in ["empirical_coverage", "average_set_size", "empty_set_proportion", "top1_accuracy"]:
    assert f"{m}_aps" in paired and f"{m}_lac" in paired
assert np.allclose(paired["top1_accuracy_aps"], paired["top1_accuracy_lac"])
paired["coverage_difference_aps_minus_lac"] = paired["empirical_coverage_aps"] - paired["empirical_coverage_lac"]
paired["set_size_difference_aps_minus_lac"] = paired["average_set_size_aps"] - paired["average_set_size_lac"]
paired["empty_rate_difference_aps_minus_lac"] = paired["empty_set_proportion_aps"] - paired["empty_set_proportion_lac"]
cols = ["empirical_coverage_aps", "coverage_gap_aps", "average_set_size_aps", "empty_set_proportion_aps", "top1_accuracy_aps", "empirical_coverage_lac", "average_set_size_lac", "empty_set_proportion_lac", "coverage_difference_aps_minus_lac", "set_size_difference_aps_minus_lac"]
summary = paired.groupby(["model_key", "condition", "severity"])[cols].agg(["mean", "std"]).reset_index()
summary.columns = ["_".join(str(x) for x in col if x != "") if isinstance(col, tuple) else col for col in summary.columns]
summary = summary.rename(columns={"model_key_":"model_key", "condition_":"condition", "severity_":"severity"})
aps_corruption_table.to_csv(RESULTS_ROOT / "aps_corruption_metrics.csv", index=False)
aps_corruption_size_distribution.to_csv(RESULTS_ROOT / "aps_corruption_set_size_distribution.csv", index=False)
paired.to_csv(RESULTS_ROOT / "aps_lac_paired_corruption_comparison.csv", index=False)
summary.to_csv(RESULTS_ROOT / "aps_lac_corruption_summary_mean_std.csv", index=False)
print("APS shifted evaluation: 54 split-specific rows; six fixed clean thresholds; no shifted recalibration or inference: PASS")
print("LAC comparison: exact one-to-one pairing on model, split, corruption, severity; top-1 accuracy matches: PASS")
print("APS per-split corruption metrics:")
show_table(aps_corruption_table[["model_key", "split_seed", "condition", "severity", "fixed_clean_q_hat", "empirical_coverage", "coverage_gap", "average_set_size", "empty_set_proportion", "top1_accuracy"]])
print("Mean and sample standard deviation across three splits:")
show_table(summary)
print("Saved APS corruption metrics, size distributions, paired LAC comparison, and mean/std summary under", RESULTS_ROOT)
compact_summary = paired.groupby(["model_key", "condition", "severity"]).agg(
    aps_coverage_mean=("empirical_coverage_aps", "mean"), aps_coverage_sd=("empirical_coverage_aps", "std"),
    aps_set_size_mean=("average_set_size_aps", "mean"), aps_set_size_sd=("average_set_size_aps", "std"),
    aps_empty_rate_mean=("empty_set_proportion_aps", "mean"), aps_empty_rate_sd=("empty_set_proportion_aps", "std"),
    lac_coverage_mean=("empirical_coverage_lac", "mean"), lac_coverage_sd=("empirical_coverage_lac", "std"),
    lac_set_size_mean=("average_set_size_lac", "mean"), lac_set_size_sd=("average_set_size_lac", "std"),
    coverage_delta_mean=("coverage_difference_aps_minus_lac", "mean"), set_size_delta_mean=("set_size_difference_aps_minus_lac", "mean"),
).reset_index()
print("Compact mean +/- sample SD over seeds 42, 43, 44:")
print(compact_summary.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

Randomization audit: saved uniforms reproduce exactly from seed 2026; one U is keyed by canonical index, independent of label/severity: PASS
APS shifted evaluation: 54 split-specific rows; six fixed clean thresholds; no shifted recalibration or inference: PASS
LAC comparison: exact one-to-one pairing on model, split, corruption, severity; top-1 accuracy matches: PASS
APS per-split corruption metrics:


,model_key,split_seed,condition,severity,fixed_clean_q_hat,empirical_coverage,coverage_gap,average_set_size,empty_set_proportion,top1_accuracy
0,standard_resnet18,42,gaussian_noise,1,0.925118,0.8106,0.0894,1.1876,0.0572,0.8012
1,standard_resnet18,42,gaussian_noise,3,0.925118,0.4992,0.4008,1.5086,0.0362,0.4104
2,standard_resnet18,42,gaussian_noise,5,0.925118,0.3572,0.5428,1.4726,0.0328,0.2686
3,standard_resnet18,42,defocus_blur,1,0.925118,0.8956,0.0044,0.9922,0.0722,0.9496
4,standard_resnet18,42,defocus_blur,3,0.925118,0.8692,0.0308,1.0752,0.0634,0.8890
5,standard_resnet18,42,defocus_blur,5,0.925118,0.6714,0.2286,1.5060,0.0392,0.5872
6,standard_resnet18,42,brightness,1,0.925118,0.8958,0.0042,0.9920,0.0722,0.9488
7,standard_resnet18,42,brightness,3,0.925118,0.8904,0.0096,1.0026,0.0712,0.9398
8,standard_resnet18,42,brightness,5,0.925118,0.8784,0.0216,1.0426,0.0684,0.9138
9,standard_resnet18,43,gaussian_noise,1,0.927148,0.8176,0.0824,1.2050,0.0498,0.7986


Mean and sample standard deviation across three splits:


,model_key,condition,severity,empirical_coverage_aps_mean,empirical_coverage_aps_std,coverage_gap_aps_mean,coverage_gap_aps_std,average_set_size_aps_mean,average_set_size_aps_std,empty_set_proportion_aps_mean,...,empirical_coverage_lac_mean,empirical_coverage_lac_std,average_set_size_lac_mean,average_set_size_lac_std,empty_set_proportion_lac_mean,empty_set_proportion_lac_std,coverage_difference_aps_minus_lac_mean,coverage_difference_aps_minus_lac_std,set_size_difference_aps_minus_lac_mean,set_size_difference_aps_minus_lac_std
0,augmix_resnet18,brightness,1,0.900867,0.009874,-0.000867,0.009874,1.030800,0.013657,0.075467,...,0.898867,0.001474,0.915267,0.000902,0.084733,0.000902,0.002000,0.011344,0.115533,0.014478
1,augmix_resnet18,brightness,3,0.899533,0.009653,0.000467,0.009653,1.043133,0.014264,0.073600,...,0.890067,0.002774,0.906667,0.003009,0.093333,0.003009,0.009467,0.012060,0.136467,0.017263
2,augmix_resnet18,brightness,5,0.893800,0.007203,0.006200,0.007203,1.076467,0.013828,0.069667,...,0.862067,0.000643,0.886267,0.000416,0.113733,0.000416,0.031733,0.007433,0.190200,0.014238
3,augmix_resnet18,defocus_blur,1,0.902600,0.010429,-0.002600,0.010429,1.028733,0.013645,0.075067,...,0.900600,0.001908,0.917333,0.001332,0.082667,0.001332,0.002000,0.012252,0.111400,0.014933
4,augmix_resnet18,defocus_blur,3,0.899533,0.009253,0.000467,0.009253,1.036333,0.015129,0.073600,...,0.891267,0.001617,0.909467,0.001922,0.090533,0.001922,0.008267,0.010790,0.126867,0.017010
5,augmix_resnet18,defocus_blur,5,0.885800,0.007922,0.014200,0.007922,1.084933,0.016614,0.069933,...,0.847733,0.002043,0.877000,0.001587,0.123000,0.001587,0.038067,0.009768,0.207933,0.018047
6,augmix_resnet18,gaussian_noise,1,0.884133,0.009253,0.015867,0.009253,1.095800,0.014666,0.069733,...,0.847067,0.001528,0.875333,0.001922,0.124667,0.001922,0.037067,0.008580,0.220467,0.014699
7,augmix_resnet18,gaussian_noise,3,0.811467,0.012714,0.088533,0.012714,1.297400,0.030909,0.053000,...,0.674067,0.002139,0.756800,0.007203,0.243200,0.007203,0.137400,0.014845,0.540600,0.038106
8,augmix_resnet18,gaussian_noise,5,0.742933,0.010944,0.157067,0.010944,1.390467,0.036516,0.046200,...,0.576667,0.005801,0.703000,0.008057,0.297000,0.008057,0.166267,0.016736,0.687467,0.044434
9,standard_resnet18,brightness,1,0.900933,0.005537,-0.000933,0.005537,0.995467,0.003972,0.066667,...,0.898133,0.004291,0.915067,0.005460,0.084933,0.005460,0.002800,0.003704,0.080400,0.003143


Saved APS corruption metrics, size distributions, paired LAC comparison, and mean/std summary under /content/drive/MyDrive/conformal_shift/results
Compact mean +/- sample SD over seeds 42, 43, 44:
        model_key      condition  severity  aps_coverage_mean  aps_coverage_sd  aps_set_size_mean  aps_set_size_sd  aps_empty_rate_mean  aps_empty_rate_sd  lac_coverage_mean  lac_coverage_sd  lac_set_size_mean  lac_set_size_sd  coverage_delta_mean  set_size_delta_mean
  augmix_resnet18     brightness         1             0.9009           0.0099             1.0308           0.0137               0.0755             0.0089             0.8989           0.0015             0.9153           0.0009               0.0020               0.1155
  augmix_resnet18     brightness         3             0.8995           0.0097             1.0431           0.0143               0.0736             0.0085             0.8901           0.0028             0.9067           0.0030               0.0095               0.1

## Stage 6 results

APS calibration used clean data only. The clean diagnostic flagged AugMix, seed 44 (coverage 0.9140); the independent threshold and score/set audit passed, and the recorded run continued with that result retained. The flag is not a nominal coverage-screen pass.

Corruption evaluation reused each model/split's clean threshold and the Stage 4 cached probabilities. It produced 54 split-specific rows and 594 set-size bins. The paired LAC table matches model, split, corruption, and severity one-to-one. Summaries report mean and sample standard deviation across three overlapping split sensitivity checks.

The completed RAPS analysis follows in Stage 7.

# Stage 7 - RAPS calibration and corruption evaluation

RAPS adds `lambda * max(rank - k_reg, 0)` to the randomized APS score, using 1-based class ranks. This experiment fixes `lambda=0.02` and `k_reg=1` before shifted evaluation. Calibration, canonical-index randomization, and the held-fixed clean thresholds follow the same split rules as APS.

In [ ]:
RAPS_LAMBDA = 0.02
RAPS_K_REG = 1
RAPS_ALPHA = 0.10
assert RAPS_ALPHA == ALPHA and RAPS_LAMBDA == 0.02 and RAPS_K_REG == 1
raps_saved_uniforms = np.load(APS_UNIFORMS_PATH, allow_pickle=False)
raps_regenerated = np.random.default_rng(APS_RANDOM_SEED).random(N_TEST)
assert raps_saved_uniforms.shape == (N_TEST,) and np.array_equal(raps_saved_uniforms, raps_regenerated)
assert np.array_equal(raps_saved_uniforms, aps_uniform_by_index)
assert np.all((raps_saved_uniforms >= 0) & (raps_saved_uniforms < 1))

def raps_candidate_scores(probabilities, uniforms, lambda_reg=RAPS_LAMBDA, k_reg=RAPS_K_REG):
    """Randomized RAPS score for each candidate, returned in class-index order."""
    p = np.asarray(probabilities, dtype=np.float64)
    u = np.asarray(uniforms, dtype=np.float64)
    if p.ndim != 2 or u.shape != (len(p),) or p.shape[1] == 0:
        raise ValueError("Expected probabilities (N,C), uniforms (N,), and C > 0")
    if not np.isfinite(p).all() or np.any(p < 0) or np.any(p > 1):
        raise ValueError("Probabilities must be finite and lie in [0,1]")
    if not np.isfinite(u).all() or np.any(u < 0) or np.any(u >= 1):
        raise ValueError("Uniforms must be finite and lie in [0,1)")
    if not np.isfinite(lambda_reg) or lambda_reg < 0:
        raise ValueError("lambda_reg must be finite and nonnegative")
    if int(k_reg) != k_reg or not 1 <= int(k_reg) <= p.shape[1]:
        raise ValueError("k_reg must be a 1-based rank in 1..C")
    order = np.argsort(-p, axis=1, kind="stable")
    sorted_p = np.take_along_axis(p, order, axis=1)
    before = np.cumsum(sorted_p, axis=1) - sorted_p
    ranks = np.broadcast_to(np.arange(1, p.shape[1] + 1), p.shape)
    score_rank = before + u[:, None] * sorted_p + lambda_reg * np.maximum(ranks - int(k_reg), 0)
    if np.any(np.diff(score_rank, axis=1) < -1e-12):
        raise AssertionError("RAPS scores must be nondecreasing in stable rank order")
    score_class = np.empty_like(score_rank)
    np.put_along_axis(score_class, order, score_rank, axis=1)
    return score_class

def raps_nonconformity_scores(probabilities, labels, uniforms, lambda_reg=RAPS_LAMBDA, k_reg=RAPS_K_REG):
    p = np.asarray(probabilities)
    y = np.asarray(labels, dtype=np.int64)
    if p.ndim != 2 or y.shape != (len(p),) or np.any(y < 0) or np.any(y >= p.shape[1]):
        raise ValueError("Expected valid probabilities (N,C) and labels (N,)")
    return raps_candidate_scores(p, uniforms, lambda_reg, k_reg)[np.arange(len(y)), y]

def raps_prediction_sets(probabilities, q_hat, uniforms, lambda_reg=RAPS_LAMBDA, k_reg=RAPS_K_REG):
    scores = raps_candidate_scores(probabilities, uniforms, lambda_reg, k_reg)
    included = scores <= q_hat
    order = np.argsort(-np.asarray(probabilities), axis=1, kind="stable")
    ranked = np.take_along_axis(included, order, axis=1)
    if np.any((~ranked[:, :-1]) & ranked[:, 1:]):
        raise AssertionError("RAPS set must be a prefix under stable probability rank")
    return included

def finite_sample_raps_threshold(probabilities, labels, calibration_indices, uniforms, alpha=RAPS_ALPHA):
    p = np.asarray(probabilities)
    y = np.asarray(labels, dtype=np.int64)
    idx = np.asarray(calibration_indices, dtype=np.int64)
    scores = raps_nonconformity_scores(p[idx], y[idx], uniforms[idx])
    n = len(idx)
    k = math.ceil((n + 1) * (1 - alpha))
    if not 1 <= k <= n:
        raise ValueError(f"Conformal order statistic k={k} is outside 1..n={n}")
    return float(np.partition(scores, k - 1)[k - 1]), k, scores

# Hand calculation for every candidate: APS=[.25,.65,.875,.975],
# penalties=[0,.02,.04,.06], so RAPS=[.25,.67,.915,1.035]. At q=.67, set={0,1}.
toy_p = np.asarray([[.50, .30, .15, .05]], dtype=np.float64)
toy_u = np.asarray([.50])
toy_scores = raps_candidate_scores(toy_p, toy_u)
toy_set = np.flatnonzero(raps_prediction_sets(toy_p, .67, toy_u)[0]).tolist()
assert np.allclose(toy_scores[0], [.25, .67, .915, 1.035]) and toy_set == [0, 1]
# Equal probability tie order remains class 0, 1, 2; 1-based penalties are 0, .02, .04.
tie_p = np.asarray([[.40, .40, .20]], dtype=np.float64)
assert np.argsort(-tie_p, axis=1, kind="stable")[0].tolist() == [0, 1, 2]
assert np.allclose(raps_candidate_scores(tie_p, np.asarray([.50]))[0], [.20, .62, .94])
# lambda=0 matches APS scores and sets on the hand example and cached examples.
assert np.allclose(raps_candidate_scores(toy_p, toy_u, lambda_reg=0)[0], [.25, .65, .875, .975], atol=1e-12, rtol=0)
assert np.array_equal(aps_prediction_sets(toy_p, .65, toy_u), raps_prediction_sets(toy_p, .65, toy_u, lambda_reg=0))
for model_key in MODEL_KEYS:
    p = np.asarray(clean_caches[model_key]["probabilities"][:128], dtype=np.float64)
    u = raps_saved_uniforms[:128]
    order = np.argsort(-p, axis=1, kind="stable")
    sorted_p = np.take_along_axis(p, order, axis=1)
    aps_rank = np.cumsum(sorted_p, axis=1) - sorted_p + u[:, None] * sorted_p
    aps_class = np.empty_like(aps_rank)
    np.put_along_axis(aps_class, order, aps_rank, axis=1)
    assert np.allclose(raps_candidate_scores(p, u, lambda_reg=0), aps_class, atol=1e-12, rtol=0)
    assert np.array_equal(raps_prediction_sets(p, .5, u, lambda_reg=0), aps_prediction_sets(p, .5, u))
print("RAPS hand score, rank penalty, set boundary and stable ties: PASS")
print("lambda=0 APS score/set equivalence on toy plus 128 cached images/model: PASS")
print("Saved seed-2026 uniforms reproduce and remain image-indexed: PASS")

RAPS hand score, rank penalty, set boundary and stable ties: PASS
lambda=0 APS score/set equivalence on toy plus 128 cached images/model: PASS
Saved seed-2026 uniforms reproduce and remain image-indexed: PASS


## Clean calibration and validation gate

Each model/split threshold is calibrated from its 5,000 clean calibration examples only. The finite-sample order statistic is k=ceil((n+1)*(1-alpha))=4501. Clean evaluation uses the disjoint 5,000-image evaluation subset and the same saved per-index random variables. Before any corruption evaluation, this section independently reproduces each threshold and checks that score <= q_hat exactly matches candidate-set membership for every clean evaluation label. The gate flags coverage outside three nominal standard errors (about 0.0127); it does not tune RAPS parameters.

In [ ]:
raps_threshold_records = []
raps_thresholds = {}
raps_clean_records = []
raps_clean_size_records = []
raps_clean_examples = []
raps_clean_audit_records = []
for model_key in MODEL_KEYS:
    cache = clean_caches[model_key]
    p_all = cache["probabilities"]
    y_all = cache["labels"]
    for seed in SPLIT_SEEDS:
        cal_idx = splits[seed]["calibration_indices"]
        eval_idx = splits[seed]["evaluation_indices"]
        q_hat, k, cal_scores = finite_sample_raps_threshold(p_all, y_all, cal_idx, raps_saved_uniforms)
        recomputed = raps_nonconformity_scores(p_all[cal_idx], y_all[cal_idx], raps_saved_uniforms[cal_idx])
        q_recomputed = float(np.partition(recomputed, k - 1)[k - 1])
        assert len(cal_idx) == 5000 and k == 4501
        assert np.isclose(q_hat, q_recomputed, rtol=0, atol=1e-12)
        raps_thresholds[(model_key, seed)] = q_hat
        raps_threshold_records.append({
            "model_key": model_key, "split_seed": seed, "n_calibration": len(cal_idx),
            "alpha": RAPS_ALPHA, "lambda": RAPS_LAMBDA, "k_reg_1_based": RAPS_K_REG,
            "order_statistic_k_1_based": k, "q_hat": q_hat,
            "calibration_score_median": float(np.median(cal_scores)),
        })
        p = p_all[eval_idx]
        y = y_all[eval_idx]
        u = raps_saved_uniforms[eval_idx]
        membership = raps_prediction_sets(p, q_hat, u)
        sizes = membership.sum(axis=1)
        covered_by_set = membership[np.arange(len(eval_idx)), y]
        covered_by_score = raps_nonconformity_scores(p, y, u) <= q_hat
        assert np.array_equal(covered_by_set, covered_by_score)
        coverage = float(covered_by_set.mean())
        raps_clean_records.append({
            "model_key": model_key, "split_seed": seed, "n_evaluation": len(eval_idx),
            "q_hat": q_hat, "lambda": RAPS_LAMBDA, "k_reg_1_based": RAPS_K_REG,
            "empirical_coverage": coverage, "coverage_gap": TARGET_COVERAGE - coverage,
            "average_set_size": float(sizes.mean()), "empty_set_count": int((sizes == 0).sum()),
            "empty_set_proportion": float((sizes == 0).mean()),
            "top1_accuracy": float((p.argmax(axis=1) == y).mean()),
        })
        raps_clean_audit_records.append({
            "model_key": model_key, "split_seed": seed, "order_statistic_k_1_based": k,
            "q_hat_recomputed_matches": True, "score_set_membership_matches": True,
            "true_label_rows_checked": len(eval_idx),
        })
        for size in range(11):
            raps_clean_size_records.append({
                "model_key": model_key, "split_seed": seed, "set_size": size,
                "count": int((sizes == size).sum()), "proportion": float((sizes == size).mean()),
            })
        for offset, canonical_index in enumerate(eval_idx[:3]):
            raps_clean_examples.append({
                "model_key": model_key, "split_seed": seed,
                "original_test_index": int(canonical_index), "true_label": int(y[offset]),
                "probabilities": np.round(p[offset], 4).tolist(), "uniform_U": float(u[offset]),
                "q_hat": q_hat, "prediction_set": np.flatnonzero(membership[offset]).tolist(),
                "true_label_covered": bool(covered_by_set[offset]),
            })

raps_threshold_table = pd.DataFrame(raps_threshold_records)
raps_clean_table = pd.DataFrame(raps_clean_records)
raps_clean_size_distribution = pd.DataFrame(raps_clean_size_records)
raps_clean_examples_table = pd.DataFrame(raps_clean_examples)
raps_clean_audit_table = pd.DataFrame(raps_clean_audit_records)
assert len(raps_threshold_table) == len(raps_clean_table) == len(raps_clean_audit_table) == 6
assert len(raps_clean_size_distribution) == 66
assert np.isfinite(raps_threshold_table["q_hat"]).all()
assert np.isfinite(raps_clean_table[["empirical_coverage", "average_set_size", "empty_set_proportion"]].to_numpy()).all()
raps_clean_tolerance = 3 * math.sqrt(TARGET_COVERAGE * (1 - TARGET_COVERAGE) / EXPECTED_SPLIT_SIZE)
within_gate = np.abs(raps_clean_table["empirical_coverage"] - TARGET_COVERAGE) <= raps_clean_tolerance
print("Six clean RAPS thresholds (lambda=0.02, k_reg=1, alpha=0.10):")
show_table(raps_threshold_table)
print("Clean validation metrics:")
show_table(raps_clean_table)
print("Set-size distribution, sizes 0 through 10:")
show_table(raps_clean_size_distribution)
print("Threshold/score-set consistency audit:")
show_table(raps_clean_audit_table)
print(f"Clean coverage gate: {int(within_gate.sum())}/6 within ±{raps_clean_tolerance:.4f} of 0.90.")
if not within_gate.all():
    raise RuntimeError("RAPS clean gate failed; stop before corruption evaluation.")
raps_threshold_table.to_csv(RESULTS_ROOT / "raps_calibration_thresholds.csv", index=False)
raps_clean_table.to_csv(RESULTS_ROOT / "raps_clean_metrics.csv", index=False)
raps_clean_size_distribution.to_csv(RESULTS_ROOT / "raps_clean_set_size_distribution.csv", index=False)
raps_clean_examples_table.to_json(RESULTS_ROOT / "raps_clean_example_prediction_sets.json", orient="records", indent=2)
raps_clean_audit_table.to_csv(RESULTS_ROOT / "raps_clean_consistency_audit.csv", index=False)
print("CLEAN GATE PASS; saved six thresholds and clean diagnostics under", RESULTS_ROOT)

Six clean RAPS thresholds (lambda=0.02, k_reg=1, alpha=0.10):


,model_key,split_seed,n_calibration,alpha,lambda,k_reg_1_based,order_statistic_k_1_based,q_hat,calibration_score_median
0,standard_resnet18,42,5000,0.1,0.02,1,4501,0.928096,0.521197
1,standard_resnet18,43,5000,0.1,0.02,1,4501,0.928015,0.528892
2,standard_resnet18,44,5000,0.1,0.02,1,4501,0.930859,0.530421
3,augmix_resnet18,42,5000,0.1,0.02,1,4501,0.912072,0.506161
4,augmix_resnet18,43,5000,0.1,0.02,1,4501,0.917170,0.508477
5,augmix_resnet18,44,5000,0.1,0.02,1,4501,0.921036,0.517384


Clean validation metrics:


,model_key,split_seed,n_evaluation,q_hat,lambda,k_reg_1_based,empirical_coverage,coverage_gap,average_set_size,empty_set_count,empty_set_proportion,top1_accuracy
0,standard_resnet18,42,5000,0.928096,0.02,1,0.8984,0.0016,0.9820,342,0.0684,0.9506
1,standard_resnet18,43,5000,0.928015,0.02,1,0.8982,0.0018,0.9790,336,0.0672,0.9518
2,standard_resnet18,44,5000,0.930859,0.02,1,0.9068,-0.0068,0.9882,290,0.0580,0.9496
3,augmix_resnet18,42,5000,0.912072,0.02,1,0.8920,0.0080,0.9992,406,0.0812,0.9530
4,augmix_resnet18,43,5000,0.917170,0.02,1,0.9018,-0.0018,1.0102,363,0.0726,0.9530
5,augmix_resnet18,44,5000,0.921036,0.02,1,0.9118,-0.0118,1.0228,316,0.0632,0.9522


Set-size distribution, sizes 0 through 10:


,model_key,split_seed,set_size,count,proportion
0,standard_resnet18,42,0,342,0.0684
1,standard_resnet18,42,1,4439,0.8878
2,standard_resnet18,42,2,188,0.0376
3,standard_resnet18,42,3,29,0.0058
4,standard_resnet18,42,4,2,0.0004
...,...,...,...,...,...
61,augmix_resnet18,44,6,0,0.0000
62,augmix_resnet18,44,7,0,0.0000
63,augmix_resnet18,44,8,0,0.0000
64,augmix_resnet18,44,9,0,0.0000


Threshold/score-set consistency audit:


,model_key,split_seed,order_statistic_k_1_based,q_hat_recomputed_matches,score_set_membership_matches,true_label_rows_checked
0,standard_resnet18,42,4501,True,True,5000
1,standard_resnet18,43,4501,True,True,5000
2,standard_resnet18,44,4501,True,True,5000
3,augmix_resnet18,42,4501,True,True,5000
4,augmix_resnet18,43,4501,True,True,5000
5,augmix_resnet18,44,4501,True,True,5000


Clean coverage gate: 6/6 within ±0.0127 of 0.90.
CLEAN GATE PASS; saved six thresholds and clean diagnostics under /content/drive/MyDrive/conformal_shift/results


## Fixed-threshold evaluation on CIFAR-10-C

The clean gate passed for all six model/split combinations. This cell applies each unchanged clean RAPS threshold to the same saved evaluation indices for Gaussian noise, defocus blur, and brightness at severities 1, 3, and 5. It loads existing Stage 4 probability caches and reuses the same seed-2026 uniform by canonical image index. It does not rerun inference or recalibrate. Outputs retain all 54 split-specific results, 594 prediction-set-size bins, mean/sample SD over the three overlapping split sensitivity checks, and a paired LAC/APS/RAPS comparison on identical model/split/condition/severity keys.

In [ ]:
# No corruption-triggered recalibration or network inference occurs in this section.
assert len(raps_clean_table) == 6 and len(raps_threshold_table) == 6
assert np.all(np.abs(raps_clean_table["empirical_coverage"] - TARGET_COVERAGE) <= raps_clean_tolerance)
assert np.array_equal(raps_saved_uniforms, np.load(APS_UNIFORMS_PATH, allow_pickle=False))

raps_corruption_records = []
raps_corruption_size_records = []
for model_key in MODEL_KEYS:
    clean_labels = clean_caches[model_key]["labels"]
    for seed in SPLIT_SEEDS:
        eval_idx = splits[seed]["evaluation_indices"]
        q_hat = raps_thresholds[(model_key, seed)]
        uniforms = raps_saved_uniforms[eval_idx]
        for corruption in CORRUPTIONS:
            for severity in SEVERITIES:
                cache = load_cache(model_key, corruption, severity)
                assert np.array_equal(cache["labels"], clean_labels)
                p = cache["probabilities"][eval_idx]
                y = cache["labels"][eval_idx]
                membership = raps_prediction_sets(p, q_hat, uniforms)
                sizes = membership.sum(axis=1)
                covered = membership[np.arange(len(eval_idx)), y]
                true_scores = raps_nonconformity_scores(p, y, uniforms)
                assert np.array_equal(covered, true_scores <= q_hat)
                coverage = float(covered.mean())
                raps_corruption_records.append({
                    "model_key": model_key, "split_seed": seed, "condition": corruption, "severity": int(severity),
                    "n_evaluation": len(eval_idx), "fixed_clean_q_hat": q_hat,
                    "lambda": RAPS_LAMBDA, "k_reg_1_based": RAPS_K_REG,
                    "empirical_coverage": coverage, "coverage_gap": TARGET_COVERAGE - coverage,
                    "average_set_size": float(sizes.mean()), "empty_set_count": int((sizes == 0).sum()),
                    "empty_set_proportion": float((sizes == 0).mean()),
                    "top1_accuracy": float((p.argmax(axis=1) == y).mean()),
                })
                for size in range(11):
                    raps_corruption_size_records.append({
                        "model_key": model_key, "split_seed": seed, "condition": corruption, "severity": int(severity),
                        "set_size": size, "count": int((sizes == size).sum()),
                        "proportion": float((sizes == size).mean()),
                    })

raps_corruption_table = pd.DataFrame(raps_corruption_records)
raps_corruption_size_distribution = pd.DataFrame(raps_corruption_size_records)
assert len(raps_corruption_table) == 54 and len(raps_corruption_size_distribution) == 594
assert (raps_corruption_table.groupby(["model_key", "condition", "severity"]).size() == 3).all()
expected_q = {(r.model_key, int(r.split_seed)): float(r.q_hat) for r in raps_threshold_table.itertuples()}
assert all(np.isclose(r.fixed_clean_q_hat, expected_q[(r.model_key, int(r.split_seed))])
           for r in raps_corruption_table.itertuples())

# Paired comparison: exact same model/split/corruption/severity rows.
keys = ["model_key", "split_seed", "condition", "severity"]
aps_table = pd.read_csv(RESULTS_ROOT / "aps_corruption_metrics.csv")
lac_table = pd.read_csv(RESULTS_ROOT / "lac_corruption_metrics.csv")
assert len(aps_table) == len(lac_table) == 54
paired = raps_corruption_table.merge(aps_table, on=keys, suffixes=("_raps", "_aps"), validate="one_to_one")
lac_metric_columns = ["n_evaluation", "empirical_coverage", "coverage_gap", "average_set_size",
                      "empty_set_count", "empty_set_proportion", "top1_accuracy"]
lac_renamed = lac_table.rename(columns={c: f"{c}_lac" for c in lac_metric_columns})
paired = paired.merge(lac_renamed, on=keys, validate="one_to_one")
assert len(paired) == 54
assert np.allclose(paired["top1_accuracy_raps"], paired["top1_accuracy_aps"])
assert np.allclose(paired["top1_accuracy_raps"], paired["top1_accuracy_lac"])
for method in ["aps", "lac"]:
    paired[f"coverage_difference_raps_minus_{method}"] = paired[f"empirical_coverage_raps"] - paired[f"empirical_coverage_{method}"]
    paired[f"set_size_difference_raps_minus_{method}"] = paired[f"average_set_size_raps"] - paired[f"average_set_size_{method}"]
    paired[f"empty_rate_difference_raps_minus_{method}"] = paired[f"empty_set_proportion_raps"] - paired[f"empty_set_proportion_{method}"]

raps_summary = raps_corruption_table.groupby(["model_key", "condition", "severity"]).agg(
    coverage_mean=("empirical_coverage", "mean"), coverage_sample_sd=("empirical_coverage", "std"),
    coverage_gap_mean=("coverage_gap", "mean"), coverage_gap_sample_sd=("coverage_gap", "std"),
    set_size_mean=("average_set_size", "mean"), set_size_sample_sd=("average_set_size", "std"),
    empty_rate_mean=("empty_set_proportion", "mean"), empty_rate_sample_sd=("empty_set_proportion", "std"),
    top1_accuracy_mean=("top1_accuracy", "mean"), top1_accuracy_sample_sd=("top1_accuracy", "std"),
).reset_index()
comparison_summary = paired.groupby(["model_key", "condition", "severity"]).agg(
    raps_coverage_mean=("empirical_coverage_raps", "mean"), raps_coverage_sd=("empirical_coverage_raps", "std"),
    aps_coverage_mean=("empirical_coverage_aps", "mean"), aps_coverage_sd=("empirical_coverage_aps", "std"),
    lac_coverage_mean=("empirical_coverage_lac", "mean"), lac_coverage_sd=("empirical_coverage_lac", "std"),
    raps_set_size_mean=("average_set_size_raps", "mean"), raps_set_size_sd=("average_set_size_raps", "std"),
    aps_set_size_mean=("average_set_size_aps", "mean"), aps_set_size_sd=("average_set_size_aps", "std"),
    lac_set_size_mean=("average_set_size_lac", "mean"), lac_set_size_sd=("average_set_size_lac", "std"),
    raps_empty_rate_mean=("empty_set_proportion_raps", "mean"), raps_empty_rate_sd=("empty_set_proportion_raps", "std"),
    aps_empty_rate_mean=("empty_set_proportion_aps", "mean"), aps_empty_rate_sd=("empty_set_proportion_aps", "std"),
    lac_empty_rate_mean=("empty_set_proportion_lac", "mean"), lac_empty_rate_sd=("empty_set_proportion_lac", "std"),
    raps_minus_aps_coverage_mean=("coverage_difference_raps_minus_aps", "mean"),
    raps_minus_aps_set_size_mean=("set_size_difference_raps_minus_aps", "mean"),
    raps_minus_lac_coverage_mean=("coverage_difference_raps_minus_lac", "mean"),
    raps_minus_lac_set_size_mean=("set_size_difference_raps_minus_lac", "mean"),
).reset_index()

raps_corruption_table.to_csv(RESULTS_ROOT / "raps_corruption_metrics.csv", index=False)
raps_corruption_size_distribution.to_csv(RESULTS_ROOT / "raps_corruption_set_size_distribution.csv", index=False)
raps_summary.to_csv(RESULTS_ROOT / "raps_corruption_summary_mean_std.csv", index=False)
paired.to_csv(RESULTS_ROOT / "raps_lac_aps_paired_corruption_comparison.csv", index=False)
comparison_summary.to_csv(RESULTS_ROOT / "raps_lac_aps_comparison_summary_mean_std.csv", index=False)
print("RAPS corruption: 54 split rows; 594 size-distribution bins; fixed clean thresholds and cached probabilities: PASS")
print("Three-method pairing: one-to-one across all 54 model/split/condition/severity keys; cached accuracy agrees: PASS")
print("RAPS per-split results:")
show_table(raps_corruption_table[["model_key", "split_seed", "condition", "severity", "fixed_clean_q_hat", "empirical_coverage", "coverage_gap", "average_set_size", "empty_set_proportion", "top1_accuracy"]])
print("RAPS mean and sample SD across the three overlapping split sensitivity checks:")
show_table(raps_summary)
print("Paired LAC / APS / RAPS mean coverage and set-size comparison:")
show_table(comparison_summary[["model_key", "condition", "severity", "raps_coverage_mean", "raps_coverage_sd", "aps_coverage_mean", "aps_coverage_sd", "lac_coverage_mean", "lac_coverage_sd", "raps_set_size_mean", "raps_set_size_sd", "aps_set_size_mean", "aps_set_size_sd", "lac_set_size_mean", "lac_set_size_sd"]])
print("Saved RAPS metrics, size distributions, sample-SD summaries and paired comparisons under", RESULTS_ROOT)

RAPS corruption: 54 split rows; 594 size-distribution bins; fixed clean thresholds and cached probabilities: PASS
Three-method pairing: one-to-one across all 54 model/split/condition/severity keys; cached accuracy agrees: PASS
RAPS per-split results:


,model_key,split_seed,condition,severity,fixed_clean_q_hat,empirical_coverage,coverage_gap,average_set_size,empty_set_proportion,top1_accuracy
0,standard_resnet18,42,gaussian_noise,1,0.928096,0.8054,0.0946,1.1388,0.0532,0.8012
1,standard_resnet18,42,gaussian_noise,3,0.928096,0.4796,0.4204,1.4010,0.0344,0.4104
2,standard_resnet18,42,gaussian_noise,5,0.928096,0.3412,0.5588,1.3732,0.0312,0.2686
3,standard_resnet18,42,defocus_blur,1,0.928096,0.8964,0.0036,0.9836,0.0678,0.9496
4,standard_resnet18,42,defocus_blur,3,0.928096,0.8662,0.0338,1.0476,0.0598,0.8890
5,standard_resnet18,42,defocus_blur,5,0.928096,0.6550,0.2450,1.3924,0.0368,0.5872
6,standard_resnet18,42,brightness,1,0.928096,0.8968,0.0032,0.9842,0.0682,0.9488
7,standard_resnet18,42,brightness,3,0.928096,0.8914,0.0086,0.9912,0.0664,0.9398
8,standard_resnet18,42,brightness,5,0.928096,0.8788,0.0212,1.0242,0.0636,0.9138
9,standard_resnet18,43,gaussian_noise,1,0.928015,0.8074,0.0926,1.1490,0.0494,0.7986


RAPS mean and sample SD across the three overlapping split sensitivity checks:


,model_key,condition,severity,coverage_mean,coverage_sample_sd,coverage_gap_mean,coverage_gap_sample_sd,set_size_mean,set_size_sample_sd,empty_rate_mean,empty_rate_sample_sd,top1_accuracy_mean,top1_accuracy_sample_sd
0,augmix_resnet18,brightness,1,0.900133,0.008603,-0.000133,0.008603,1.011333,0.010830,0.072400,0.008400,0.951000,0.001039
1,augmix_resnet18,brightness,3,0.899533,0.008932,0.000467,0.008932,1.024067,0.010599,0.070867,0.008003,0.944000,0.001744
2,augmix_resnet18,brightness,5,0.891467,0.007726,0.008533,0.007726,1.050267,0.010801,0.067067,0.007414,0.927467,0.001102
3,augmix_resnet18,defocus_blur,1,0.901333,0.009246,-0.001333,0.009246,1.010667,0.010265,0.072200,0.008807,0.952600,0.000346
4,augmix_resnet18,defocus_blur,3,0.899067,0.009322,0.000933,0.009322,1.019267,0.012235,0.070733,0.008905,0.944933,0.000577
5,augmix_resnet18,defocus_blur,5,0.883667,0.007814,0.016333,0.007814,1.056133,0.012548,0.066667,0.007726,0.915400,0.002163
6,augmix_resnet18,gaussian_noise,1,0.881600,0.008743,0.018400,0.008743,1.067333,0.011789,0.067333,0.007401,0.914867,0.001007
7,augmix_resnet18,gaussian_noise,3,0.803000,0.011049,0.097000,0.011049,1.232333,0.021366,0.050400,0.008249,0.785000,0.001510
8,augmix_resnet18,gaussian_noise,5,0.728400,0.009824,0.171600,0.009824,1.305600,0.026595,0.044533,0.007976,0.693867,0.004310
9,standard_resnet18,brightness,1,0.899600,0.005024,0.000400,0.005024,0.984333,0.004202,0.064467,0.005460,0.949533,0.000808


Paired LAC / APS / RAPS mean coverage and set-size comparison:


,model_key,condition,severity,raps_coverage_mean,raps_coverage_sd,aps_coverage_mean,aps_coverage_sd,lac_coverage_mean,lac_coverage_sd,raps_set_size_mean,raps_set_size_sd,aps_set_size_mean,aps_set_size_sd,lac_set_size_mean,lac_set_size_sd
0,augmix_resnet18,brightness,1,0.900133,0.008603,0.900867,0.009874,0.898867,0.001474,1.011333,0.010830,1.030800,0.013657,0.915267,0.000902
1,augmix_resnet18,brightness,3,0.899533,0.008932,0.899533,0.009653,0.890067,0.002774,1.024067,0.010599,1.043133,0.014264,0.906667,0.003009
2,augmix_resnet18,brightness,5,0.891467,0.007726,0.893800,0.007203,0.862067,0.000643,1.050267,0.010801,1.076467,0.013828,0.886267,0.000416
3,augmix_resnet18,defocus_blur,1,0.901333,0.009246,0.902600,0.010429,0.900600,0.001908,1.010667,0.010265,1.028733,0.013645,0.917333,0.001332
4,augmix_resnet18,defocus_blur,3,0.899067,0.009322,0.899533,0.009253,0.891267,0.001617,1.019267,0.012235,1.036333,0.015129,0.909467,0.001922
5,augmix_resnet18,defocus_blur,5,0.883667,0.007814,0.885800,0.007922,0.847733,0.002043,1.056133,0.012548,1.084933,0.016614,0.877000,0.001587
6,augmix_resnet18,gaussian_noise,1,0.881600,0.008743,0.884133,0.009253,0.847067,0.001528,1.067333,0.011789,1.095800,0.014666,0.875333,0.001922
7,augmix_resnet18,gaussian_noise,3,0.803000,0.011049,0.811467,0.012714,0.674067,0.002139,1.232333,0.021366,1.297400,0.030909,0.756800,0.007203
8,augmix_resnet18,gaussian_noise,5,0.728400,0.009824,0.742933,0.010944,0.576667,0.005801,1.305600,0.026595,1.390467,0.036516,0.703000,0.008057
9,standard_resnet18,brightness,1,0.899600,0.005024,0.900933,0.005537,0.898133,0.004291,0.984333,0.004202,0.995467,0.003972,0.915067,0.005460


Saved RAPS metrics, size distributions, sample-SD summaries and paired comparisons under /content/drive/MyDrive/conformal_shift/results


## Stage 7 results

RAPS used the fixed `lambda=0.02`, `k_reg=1`, and `alpha=0.10`. The hand calculation, stable tie handling, 1-based rank penalties, saved seed-2026 randomization, lambda=0 APS equivalence, six threshold reproductions, and all 30,000 clean true-label score/set checks passed. All six RAPS clean coverages were within the 0.0127 diagnostic tolerance.

Corruption evaluation produced 54 split-specific rows and 594 set-size bins from cached probabilities. Every split kept its clean threshold fixed. The LAC/APS/RAPS comparison matches model, split, condition, and severity one-to-one, with identical cached top-1 accuracy. The three overlapping splits are sensitivity checks; sample SD is descriptive.

Stage 8 is completed in `04_label_shift_experiments.ipynb`, which uses these saved clean thresholds and the clean prediction caches for class-conditional and label-shift analysis.